In [3]:
import pandas as pd

In [28]:
df = pd.read_csv('../data/raw/nyc_sep.csv', encoding='utf-8')
df.shape
df.head(5)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1352055619826961511,https://www.airbnb.com/rooms/1352055619826961511,20260914002229,2026-09-14,NaN,"New 3 bed, 1.5 bath",This stylish place to stay is perfect for grou...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,677651543,...,NaN,NaN,NaN,NaN,NaN,1,1,0,0,NaN
1,1215930899551057941,https://www.airbnb.com/rooms/1215930899551057941,20260914002229,2026-09-14,NaN,Queens apartament,Bring the whole family to this great place wit...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,392517036,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
2,1203866180302079266,https://www.airbnb.com/rooms/1203866180302079266,20260914002229,2026-09-15,NaN,Luxury Meets Comfort Stay NYC,This stylish place is close to must-see destin...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,115082911,...,NaN,NaN,NaN,NaN,NaN,2,0,2,0,NaN
3,15310580,https://www.airbnb.com/rooms/15310580,20260914002229,2026-09-15,previous scrape,Nice apartment in Gramercy.,Nice apartment in one of the best neighbors of NY,NaN,https://a0.muscache.com/pictures/c2c78aad-78c9...,27336207,...,4.88,5.00,4.25,NaN,NaN,2,2,0,0,0.07
4,32842295,https://www.airbnb.com/rooms/32842295,20260914002229,2026-09-15,previous scrape,Beautiful bedroom and full bathroom,NaN,NaN,https://a0.muscache.com/pictures/b946d6e3-07d6...,233227636,...,4.89,4.78,4.75,NaN,NaN,1,0,1,0,0.40


In [29]:
df.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_profile_id', 'host_profile_url', 'host_name',
       'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months',
       'hosts_time_as_host_years', 'hosts_time_as_host_months',
       'host_location', 'host_about', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_is_superhost',
       'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood',
       'host_listings_count', 'host_total_listings_count',
       'host_verifications', 'host_has_profile_pic', 'host_identity_verified',
       'neighbourhood', 'neighbourhood_cleansed',
       'neighbourhood_group_cleansed', 'latitude', 'longitude',
       'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'price_quote_checkin_date', 'price_quote_c

In [30]:
df.shape

(29918, 90)

## Requirements for data cleaning
`host_identity_verified` should be t,
`price_quote_price_per_night` should not be NA,
`has_availability` should be t,
active referring to `estimated_occupancy_l365d` not 0 or `availability_365` (availability for next 365 days) not 0




In [31]:
verified = df['host_identity_verified'] == 't'
has_price = df['price_quote_price_per_night'].notna()
available = df['has_availability'] == 't'
active = ((df['estimated_occupancy_l365d'] != 0) | (df['availability_365'] != 0))

df_clean = df[verified & has_price & available & active]
df_clean.shape

(18460, 90)

In [32]:
df_clean.loc[df_clean['maximum_nights'] < df_clean['minimum_nights'], ['accommodates', 'beds', 'bedrooms']]

,accommodates,beds,bedrooms


In [33]:
df_clean['review_scores_rating'].describe()

count    13661.000000
mean         4.724584
std          0.469595
min          1.000000
25%          4.670000
50%          4.860000
75%          5.000000
max          5.000000
Name: review_scores_rating, dtype: float64

In [38]:
df_analysis = df_clean[['id', 'name', 'host_id', 'host_name', 'host_is_superhost', 'host_listings_count', 'neighbourhood_cleansed', 
    'neighbourhood_group_cleansed', 'latitude', 'longitude', 'room_type', 'accommodates', 'price_quote_checkin_date', 'price_quote_checkout_date',
    'price_quote_price_per_night', 'minimum_nights', 'number_of_reviews', 'number_of_reviews_l30d', 'review_scores_rating', 
    'reviews_per_month', 'availability_30', 'calendar_last_scraped', 'calculated_host_listings_count', 'estimated_occupancy_l365d', 
    'estimated_revenue_l365d']]

df_analysis.shape


(18460, 25)

In [35]:
df_analysis['price_quote_checkin_date'] = pd.to_datetime(df_analysis['price_quote_checkin_date'], format='%Y-%m-%d')
df_analysis['price_quote_checkin_date'].head(5)

1    2026-09-15
7    2026-09-14
8    2026-09-14
9    2026-09-15
10   2026-09-15
Name: price_quote_checkin_date, dtype: datetime64[us]

In [36]:
# Creating a week aggregation
df_analysis['checkin_week'] = df_analysis['price_quote_checkin_date'].dt.to_period('W').dt.start_time
df_analysis['checkin_week'].value_counts().sort_values(ascending=False).head(20)

checkin_week
2026-09-14    6431
2026-09-28    1911
2026-09-21    1426
2026-10-26    1145
2026-12-28     887
2026-10-05     739
2026-10-12     732
2026-11-02     660
2026-11-30     588
2026-10-19     465
2026-12-14     420
2026-11-09     342
2026-11-16     275
2026-12-21     263
2026-11-23     231
2026-12-07     193
2027-01-04     193
2027-02-01     151
2027-03-29     139
2027-05-31     122
Name: count, dtype: int64

In [23]:
# creating a function for week aggregation:
def get_week(df, date_col = 'price_quote_checkin_date'):
    df['checkin_week'] = pd.to_datetime(df[date_col]).dt.to_period('W').dt.start_time

In [37]:
df_analysis.groupby('checkin_week').agg(
    ave_price=('price_quote_price_per_night', 'mean')
).sort_values(by='checkin_week', ascending=True).head(20)

,ave_price
checkin_week,
2026-09-14,407.002723
2026-09-21,458.997973
2026-09-28,222.671544
2026-10-05,203.118390
2026-10-12,230.857664
2026-10-19,230.916796
2026-10-26,195.303336
2026-11-02,238.169470
2026-11-09,213.405146


In [39]:
df_analysis['price_quote_per_night_per_person'] = df_analysis['price_quote_price_per_night'] / df_analysis['accommodates']
df_analysis['price_quote_per_night_per_person'].describe()

count    18460.000000
mean       108.219141
std        184.581222
min          4.178750
25%         46.624583
50%         70.956667
75%        116.273750
max      11028.870000
Name: price_quote_per_night_per_person, dtype: float64

In [51]:
df_analysis.groupby(['neighbourhood_group_cleansed'])['price_quote_per_night_per_person'].describe()

,count,mean,std,min,25%,50%,75%,max
neighbourhood_group_cleansed,,,,,,,,
Bronx,755.0,73.695915,401.594889,8.021875,35.52000,48.200000,69.1075,11028.870
Brooklyn,6255.0,83.791600,96.659286,4.178750,41.97250,63.911667,100.0000,4221.850
Manhattan,7994.0,146.342523,212.488262,7.067500,61.37375,91.402500,160.8125,6139.535
Queens,3173.0,72.731757,144.807800,6.241250,36.19500,54.190000,78.8450,5000.000
Staten Island,283.0,61.231361,45.627534,13.633333,33.90500,49.800000,69.8250,406.500
